In [ ]:
from mg_sampling import PROJECT_DIR
import seaborn as sns
import pandas as pd
from ast import literal_eval
import itertools

In [ ]:
systems = ['7ez0', '7ez0_noion', '7ez0_2', '7ez0_noion_2']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p', 'ecc_spce']
repids = [0, 11]

df = []
for system, forcefield, repid in itertools.product(systems, forcefields, repids):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{repid:02d}'
    _df = pd.read_csv(path / 'ion_analysis.csv', na_filter=False, converters={'inner_names': literal_eval, 'inner_resids': literal_eval, 'inner_resnames': literal_eval, 'inner_indices': literal_eval})
    _df = _df.loc[_df['time'] % 1000.0 == 0]
    _df['system'] = system.removesuffix('_2')
    _df['forcefield'] = forcefield
    _df['repid'] = repid
    _df['replicate'] = 1 if system.endswith('_2') else 0
    df.append(_df)
df = pd.concat(df)
for col in ['name', 'mgrna', 'forcefield', 'system']:
    df[col] = df[col].astype('category')

ref = pd.read_csv(PROJECT_DIR / 'structures' / '7ez0' / 'ion_analysis.csv', na_filter=False, converters={'inner_names': literal_eval, 'inner_resids': literal_eval, 'inner_resnames': literal_eval, 'inner_indices': literal_eval})
ref['system'] = '7ez0'
ref['forcefield'] = 'PDB'
ref['repid'] = 0
ref['replicate'] = 0

df = pd.concat([df, ref])


In [ ]:
df.head()

In [ ]:
hue_order = ['PDB']+forcefields
palette = ['#111111'] + sns.color_palette().as_hex()[:4]

In [ ]:
palette

In [ ]:
tstart = 100000

grouped = df.query(f"time >= {tstart} or forcefield == 'PDB'").groupby(['time', 'name', 'n_inner', 'forcefield', 'system', 'replicate'], observed=False).size().reset_index(name='counts')

g = sns.FacetGrid(grouped, row='name', col='system')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='n_inner',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
    orient='v',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

In [ ]:
grouped = df.query(f"time >= {tstart} or forcefield == 'PDB'")[['time', 'name', 'inner_names', 'forcefield', 'system', 'replicate']]
grouped = grouped.explode('inner_names').groupby(['time', 'name', 'inner_names', 'forcefield', 'system', 'replicate'], observed=False).size().reset_index(name='counts')

g = sns.FacetGrid(grouped, row='name', col='system', aspect=3, row_order=['MG', 'NA'])
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='inner_names',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
    orient='v',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"Site (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

In [ ]:
grouped = df.query(f"(time >= {tstart} or forcefield == 'PDB') and n_inner > 0")[['time', 'name', 'mgrna', 'forcefield', 'system', 'replicate']]
grouped = grouped.groupby(['time', 'name', 'mgrna', 'forcefield', 'system', 'replicate'], observed=False).size().reset_index(name='counts')

mgrna = grouped.groupby('mgrna').size()
keep = mgrna[mgrna > 2000].index
grouped['mgrna'] = grouped['mgrna'].apply(lambda x: x if x in keep else 'other')

In [ ]:
g = sns.FacetGrid(grouped, row='name', col='system', aspect=2, row_order=['MG', 'NA'], sharex='row', sharey='row')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='mgrna',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
    orient='v',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"Site (inner-sphere)")
g.set_titles("{row_name} ({col_name})")
g.tick_params(axis='x', labelbottom=True, labelrotation=45)
g.fig.tight_layout()

### Time series

In [ ]:
grouped = df.query("name == 'MG' and forcefield != 'PDB' and time % 1000 == 0").groupby(['time', 'n_inner', 'system', 'replicate', 'forcefield'], observed=False).size().reset_index(name='counts')
grouped.set_index('time')

g = sns.FacetGrid(grouped, row='n_inner', col='system')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.lineplot,
    x='time',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles(r"$n_\text{inner} =$ {row_name} ({col_name})")

Already at the beginning there are inner-sphere bound MG… Thanks to genion??

In [ ]:
import MDAnalysis as mda
u = mda.Universe(PROJECT_DIR / 'structures' / '7ez0' / '7ez0.gro')
n_resnames = pd.DataFrame({'inner_resnames': u.select_atoms('nucleic').residues.resnames})
n_resnames = n_resnames.groupby('inner_resnames').size()

grouped = df.query(f"(time >= {tstart} or forcefield == 'PDB') and n_inner > 0")[['time', 'name', 'inner_resnames', 'forcefield', 'system', 'replicate']]
grouped = grouped.explode('inner_resnames').query("inner_resnames in ['A', 'C', 'G', 'U']").groupby(['time', 'name', 'inner_resnames', 'forcefield', 'system', 'replicate'], observed=False).size()
grouped = grouped / n_resnames
grouped = grouped.reset_index(name='counts')

In [ ]:
g = sns.FacetGrid(grouped, row='name', col='system', aspect=3, row_order=['MG', 'NA', 'CL'])
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='inner_resnames',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
    orient='v',
)

g.add_legend(g._legend_data)
g.set_ylabels(r"$n_\text{bonds} / n_\text{base}$")
g.set_xlabels(r"Site (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

Check whether reference binding sites are recovered…

In [ ]:
ref = df.query("name == 'MG' and forcefield == 'PDB' and n_inner > 0")['inner_indices'].explode()

In [ ]:
grouped = df.query("name == 'MG' and forcefield != 'PDB' and n_inner > 0 and time % 1000 == 0")[['time', 'name', 'inner_indices', 'forcefield', 'system', 'replicate']]
grouped = grouped.explode('inner_indices').set_index(['time', 'name', 'forcefield', 'system', 'replicate'])
grouped = grouped.apply(lambda x: np.isin(x, ref))

In [ ]:
grouped = (grouped.groupby(grouped.index.names).sum() / len(ref) * 100).reset_index()

In [ ]:
g = sns.FacetGrid(grouped, col='system')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.lineplot,
    x='time',
    y='inner_indices',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
)

g.add_legend(g._legend_data)
g.set_ylabels("Native bonds [%]")
g.set_xlabels(r"Time [ps]")

In [ ]:
import numpy as np
systems = ['7ez0', '7ez0_noion', '7ez0_2', '7ez0_noion_2']
forcefields = ['mMg_opc', 'mMg_tip3p', 'nMg_tip3p', 'ecc_spce']
repdf = []
for system in systems:
    for forcefield in forcefields:
        _reptemp = np.loadtxt(PROJECT_DIR / 'replex' / system / forcefield / 'replica_temp.xvg')
        for i in range(_reptemp.shape[1] - 1):
            repdf.append({
                'system': system.removesuffix('_2'),
                'forcefield': forcefield,
                'replicate': 1 if system.endswith('_2') else 0,
                'repid': i,
                'time': _reptemp[::1000, 0],
                'reptemp': _reptemp[::1000, i+1],
            })
repdf = pd.DataFrame(repdf)

In [ ]:
repdf = repdf.explode(['time', 'reptemp'], ignore_index=True)

In [ ]:
grouped = repdf.groupby(['system', 'forcefield', 'replicate', 'repid', 'reptemp'], observed=False).size()
grouped = grouped.reset_index(name='counts')
grouped = grouped.set_index(['system', 'forcefield', 'replicate'])
grouped.index = grouped.index.to_flat_index()
grouped = grouped.reset_index()
grouped

In [ ]:
g = sns.FacetGrid(grouped.query("repid in [0, 4, 8, 11]"), row='index', col='repid')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='reptemp',
    y='counts',
)

g.add_legend(g._legend_data)
g.set_ylabels("Frames")
g.set_xlabels(r"Time [ps]")

In [ ]:
for replicate in [0, 1]:
    g = sns.FacetGrid(repdf.query(f'replicate == {replicate}'), col='system', row='forcefield', aspect=2)
    sns.set_style("ticks", {'axes.grid': True})

    g.map_dataframe(
        sns.lineplot,
        x='time',
        y='reptemp',
        hue='repid',
        palette='rainbow',
    )

    g.add_legend(g._legend_data)
    g.set_ylabels("Replica temp.")
    g.set_xlabels(r"Time [ps]")
    plt.show()

In [ ]:
tstart = 100000

grouped = df.query(f"system == '7ez0' and time >= {tstart} or forcefield == 'PDB'").groupby(['time', 'name', 'n_inner', 'forcefield', 'system', 'replicate', 'repid'], observed=False).size().reset_index(name='counts')

g = sns.FacetGrid(grouped, row='name', col='repid')
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.barplot,
    x='n_inner',
    y='counts',
    hue='forcefield',
    hue_order=hue_order,
    palette=palette,
    orient='v',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} (replica {col_name})")

In [ ]:
from mg_sampling.tools import read_plumed
systems = ['7ez0', '7ez0_noion']
repids = [0, 6, 11]
colvars = []
for system, forcefield, repid in itertools.product(systems, forcefields, repids):
    path = PROJECT_DIR / 'replex' / system / forcefield / f'rep{repid:02d}'
    colvar = read_plumed(path / 'cv.COLVAR')
    colvar['system'] = system
    colvar['forcefield'] = forcefield
    colvar['repid'] = repid
    colvars.append(colvar)
colvars = pd.concat(colvars)

In [ ]:
colvars

In [ ]:
colvars['cv.nb'] = colvars[filter(lambda c: c.startswith('cv.nb'), colvars.columns)].agg(list, axis=1)
colvars['cv.nc'] = colvars[filter(lambda c: c.startswith('cv.nc'), colvars.columns)].agg(list, axis=1)
colvars = colvars.drop(columns=filter(lambda c: c.startswith('cv.nb-') or c.startswith('cv.nc-'), colvars.columns))

In [ ]:
colvars = colvars.explode(['cv.nb', 'cv.nc'], ignore_index=True)

In [ ]:
colvars

In [ ]:
from matplotlib.colors import LogNorm
g = sns.FacetGrid(colvars.query("system == '7ez0_noion'"), row='forcefield', col='repid')

g.map_dataframe(
    sns.histplot,
    x='cv.nb',
    y='cv.nc',
    hue_norm=LogNorm(),
    binwidth=0.1,
)

In [ ]:
from matplotlib.colors import LogNorm
g = sns.FacetGrid(colvars.query("system == '7ez0_noion'"), row='forcefield', col='repid')

g.map_dataframe(
    sns.histplot,
    x='cv.nb',
    binwidth=0.1,
    stat='density',
)

In [ ]:
# We wanna reweight with this additional bias
def ramp(nop, a):
    return a * nop

